# Espaços de busca dos modelos de Ensemble — Random Forest e Gradient Boosting (#186)

Este notebook documenta e justifica os espaços de busca aleatória (`RandomizedSearchCV`) do Random Forest e do Gradient Boosting, definidos em `src/ensembles.py`. Ele **não executa nenhuma busca**: treinar as combinações nos folds do #102 e comparar contra os pisos fica para os cards que dependem deste — #187 (Random Forest), #188 e #189 (Gradient Boosting) e #190.

**Pontos de partida em comum com o candidato único da seção 4.3.** A base analítica tem 484.915 registros e 20,44% de Detratores (seção 4.2.1 da documentação); o treino usado pela validação cruzada por Cliente do #102, depois do split, tem 341.962 linhas. O contrato de features do card 01 (`FEATURE_SET_V1`, em `scripts/preprocessamento_nps.py`) declara 11 features brutas, e é o mesmo contrato usado aqui.

**Gradient Boosting: duas bibliotecas, por enquanto.** O #185 ainda não decidiu entre `HistGradientBoostingClassifier` (scikit-learn) e `XGBClassifier` (XGBoost) para o trabalho de Ensemble, então este notebook e `src/ensembles.py` mantêm os dois espaços lado a lado. No dia em que o #185 decidir, a biblioteca descartada sai dos dois lugares no mesmo commit.

**Os testes que garantem que os espaços funcionam** ficam em `tests/test_ensembles.py`, e não neste notebook: eles sorteiam 40 combinações de cada espaço com semente fixa e instanciam o estimador correspondente, o que é rápido demais para justificar rodar aqui a cada execução do notebook. Este notebook sorteia só uma amostra pequena, para inspeção visual dos valores.

## 1. Random Forest

### 1.1. Espaço de busca declarado

A célula abaixo importa `ESPACO_RANDOM_FOREST` de `src/ensembles.py` e lista os cinco eixos declarados. A importação do módulo não ajusta nenhum estimador (CR01): o que a célula imprime é a distribuição em si, não uma amostra dela.

In [ ]:
import sys
from pathlib import Path

for pasta_codigo in [Path("src"), Path("../src"), Path("/content/src")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break

from ensembles import ESPACO_RANDOM_FOREST


def descrever(distribuicao):
    """Nome e parâmetros da distribuição, em vez do endereço de memória do repr padrão.

    O repr padrão de um objeto `scipy.stats` traz o endereço em memória, que muda
    a cada execução e sujaria o diff do notebook sem nenhuma informação nova.
    """
    if isinstance(distribuicao, list):
        return f"lista com {len(distribuicao)} opções: {distribuicao}"
    return f"scipy.stats.{distribuicao.dist.name}, parâmetros={distribuicao.args or distribuicao.kwds}"


for nome, distribuicao in ESPACO_RANDOM_FOREST.items():
    print(f"{nome}: {descrever(distribuicao)}")

**Leitura da saída.** Cinco eixos: `n_estimators`, `max_depth` e `min_samples_leaf` são inteiros sorteados por `scipy.stats.randint`; `max_features` é uma fração contínua sorteada por `scipy.stats.uniform`; `class_weight` é a única lista simples do espaço, porque reponderação de classe é uma escolha categórica, sem escala, e não uma distribuição numérica. A tabela consolidada da seção 3 traz o intervalo e a justificativa de cada um.

### 1.2. Amostra de combinações, para inspeção visual

`ParameterSampler` sorteia cinco combinações do espaço com semente fixa. Cinco é o suficiente para olhar os valores a olho nu; a garantia de que 40 combinações são aceitas pelo `RandomForestClassifier` é o que `tests/test_ensembles.py` cobre, não esta célula.

In [ ]:
import pandas as pd
from sklearn.model_selection import ParameterSampler

SEMENTE = 42

amostra_rf = pd.DataFrame(
    ParameterSampler(ESPACO_RANDOM_FOREST, n_iter=5, random_state=SEMENTE)
)
amostra_rf

**Leitura da tabela.** Cada linha é uma combinação que um `RandomizedSearchCV` real sortearia e passaria ao `RandomForestClassifier`. `max_features` sai como fração entre 0,3 e 1,0, nunca uma contagem de colunas; `class_weight` alterna entre as três opções declaradas. Os valores concretos mudam se a semente mudar, mas a mesma semente reproduz sempre esta tabela, como `test_amostragem_e_reprodutivel_com_a_mesma_semente` confere.

## 2. Gradient Boosting

Duas bibliotecas convivem aqui até o #185 decidir. As duas seções abaixo seguem a mesma estrutura da seção 1: espaço declarado, depois amostra pequena para inspeção.

### 2.1. `HistGradientBoostingClassifier` (scikit-learn)

Mesma biblioteca do candidato único da seção 4.3 do notebook `modelagem.ipynb`. A célula lista `ESPACO_GRADIENT_BOOSTING_HISTGB`.

In [ ]:
from ensembles import ESPACO_GRADIENT_BOOSTING_HISTGB

for nome, distribuicao in ESPACO_GRADIENT_BOOSTING_HISTGB.items():
    print(f"{nome}: {descrever(distribuicao)}")

**Leitura da saída.** Cinco eixos: `learning_rate` é log-uniforme, porque o efeito do passo sobre o número de iterações necessário é multiplicativo; `max_leaf_nodes`, `max_iter` e `min_samples_leaf` são inteiros; `class_weight` é lista, pelo mesmo motivo da seção 1. `early_stopping` não entra no espaço — continua fixo em `False` em `criar_candidato` (`src/modelo.py`), porque ligá-lo vazaria Cliente entre ajuste e medição interna, e isso não é uma escolha que a busca deva reabrir.

### 2.2. `XGBClassifier` (XGBoost)

A célula lista `ESPACO_GRADIENT_BOOSTING_XGBOOST`. Sete eixos, dois a mais que o espaço do HistGB, porque o XGBoost expõe sub-amostragem de linhas e de colunas como parâmetros próprios, que o HistGB não tem por discretizar em histogramas em vez de amostrar.

In [ ]:
from ensembles import ESPACO_GRADIENT_BOOSTING_XGBOOST

for nome, distribuicao in ESPACO_GRADIENT_BOOSTING_XGBOOST.items():
    print(f"{nome}: {descrever(distribuicao)}")

**Leitura da saída.** `learning_rate` usa a mesma distribuição log-uniforme do HistGB, para o custo por ajuste do #185 comparar as duas bibliotecas sob o mesmo raciocínio de passo. `scale_pos_weight` é o único eixo com teto calculado a partir da base: `RAZAO_DESBALANCEAMENTO` (~3,89) é a proporção de não-Detratores por Detrator, e um `scale_pos_weight` nesse valor neutraliza o desbalanceamento por completo.

### 2.3. Amostra combinada, para inspeção visual

Uma amostra pequena de cada biblioteca, lado a lado.

In [ ]:
amostra_histgb = pd.DataFrame(
    ParameterSampler(ESPACO_GRADIENT_BOOSTING_HISTGB, n_iter=5, random_state=SEMENTE)
)
amostra_xgboost = pd.DataFrame(
    ParameterSampler(ESPACO_GRADIENT_BOOSTING_XGBOOST, n_iter=5, random_state=SEMENTE)
)
print("HistGradientBoostingClassifier:")
display(amostra_histgb)
print("XGBClassifier:")
display(amostra_xgboost)

**Leitura das tabelas.** As duas amostram a mesma semente e o mesmo `n_iter`, mas eixos diferentes: `max_leaf_nodes` de um lado, `max_depth` do outro, porque cada biblioteca limita o tamanho da árvore à sua maneira. Nenhuma das duas tabelas depende da outra ter rodado antes — os dois espaços são independentes, e é isso que permite descartar um deles no dia em que o #185 decidir sem tocar no que sobra.

## 3. Tabela consolidada de hiperparâmetros (CR04)

| Modelo | Hiperparâmetro | Distribuição | Intervalo | Justificativa |
|---|---|---|---|---|
| Random Forest | `n_estimators` | `randint` | 200 a 600 | Com 341.962 linhas de treino a variância de cada árvore já é pequena; acima de algumas centenas o ganho de agregar mais árvores fica marginal e o custo segue linear no número delas. |
| Random Forest | `max_depth` | `randint` | 3 a 20 | Teto próximo de log₂(341.962) ≈ 18,4: uma árvore balanceada raramente precisa de mais níveis para separar o treino inteiro. Profundidade ilimitada arriscaria árvores memorizando Cliente, o mesmo risco documentado para `min_samples_leaf` no #103. |
| Random Forest | `min_samples_leaf` | `randint` | 1 a 100 | 100 é o valor que o #103 escolheu para o Gradient Boosting nesta mesma base; o piso em 1 mantém a ponta não regularizada disponível para a busca comparar. |
| Random Forest | `max_features` | `uniform` | 0,3 a 1,0 (fração) | O contrato do card 01 declara 11 features brutas; `"sqrt"`/`"log2"` datam de bases com centenas de colunas e amostrariam ~3 delas por split, descartando a maior parte do sinal disponível. |
| Random Forest | `class_weight` | lista | `balanced`, `balanced_subsample`, `None` | 20,44% de detração é desbalanceamento moderado. Ao contrário do candidato único (onde reponderar prejudicaria a probabilidade calibrada do #103), o Random Forest vota por árvore e não expõe a mesma dependência direta da calibração — a busca decide empiricamente qual das três funciona melhor nesta base. |
| Gradient Boosting (HistGB) | `learning_rate` | `loguniform` | 0,01 a 0,3 | O #103 fixou 0,05 e a grade do #104 testou 0,05/0,10; escala log porque o efeito do passo sobre o número de iterações necessário é multiplicativo. |
| Gradient Boosting (HistGB) | `max_leaf_nodes` | `randint` | 15 a 127 | Cobre o padrão da biblioteca (31, usado pelo #103) e o ponto testado pela grade do #104 (63), com margem para os dois lados. |
| Gradient Boosting (HistGB) | `max_iter` | `randint` | 100 a 600 | Cobre os três pontos da grade do #104 (150, 300, 600). Sem parada antecipada, é o único limite do ensemble. |
| Gradient Boosting (HistGB) | `l2_regularization` | `uniform` | 0,0 a 2,0 | O #103 fixou 1,0 contra o padrão 0,0 da biblioteca; o intervalo cobre os dois extremos e o dobro da escolha do #103. |
| Gradient Boosting (HistGB) | `min_samples_leaf` | `randint` | 20 a 200 | Mesmo raciocínio do Random Forest: folha pequena sobre 341.962 linhas memoriza Cliente. O piso de 20 é o padrão da biblioteca. |
| Gradient Boosting (HistGB) | `class_weight` | lista | `None`, `balanced` | O #103 documenta por que `class_weight=None` foi a escolha do candidato: reponderar deslocaria a probabilidade predita, piorando o Brier da seção 6 e o limiar da seção 7. O eixo entra na busca para que a comparação empírica sustente essa escolha, não só o argumento teórico. |
| Gradient Boosting (XGBoost) | `learning_rate` | `loguniform` | 0,01 a 0,3 | Mesma faixa e mesma razão log-uniforme do HistGB, por serem as duas gradient boosting sobre árvores. |
| Gradient Boosting (XGBoost) | `max_depth` | `randint` | 3 a 10 | O XGBoost limita a árvore por profundidade, e não por número de folhas; 10 níveis já produzem até 2¹⁰ folhas, ordem de interação comparável ao teto de 127 folhas do espaço irmão. |
| Gradient Boosting (XGBoost) | `n_estimators` | `randint` | 100 a 600 | Mesmo intervalo de `max_iter` do HistGB, para que o custo por ajuste medido no #185 seja comparável entre as duas bibliotecas sob o mesmo orçamento de iterações. |
| Gradient Boosting (XGBoost) | `min_child_weight` | `randint` | 1 a 20 | Equivalente do XGBoost a `min_samples_leaf`: soma mínima de peso das observações numa folha antes dela poder ser criada, mesma lógica de regularização contra folha memorizando Cliente. |
| Gradient Boosting (XGBoost) | `reg_lambda` | `uniform` | 0,0 a 2,0 | Mesma faixa de `l2_regularization` do HistGB, pelo nome equivalente no XGBoost. |
| Gradient Boosting (XGBoost) | `subsample` | `uniform` | 0,6 a 1,0 | Mecanismo de regularização do XGBoost sem par direto no HistGB (que discretiza em histogramas em vez de amostrar linhas). O piso de 0,6 evita perder informação suficiente para exigir muito mais iterações. |
| Gradient Boosting (XGBoost) | `colsample_bytree` | `uniform` | 0,5 a 1,0 | Equivalente a `max_features` do Random Forest: com 11 features brutas no contrato do card 01, um piso baixo deixaria cada árvore enxergar poucas colunas. |
| Gradient Boosting (XGBoost) | `scale_pos_weight` | `uniform` | 1,0 a ~3,89 | O teto é `RAZAO_DESBALANCEAMENTO`, a proporção observada de não-Detratores por Detrator (385.775 / 99.140). O XGBoost não tem uma versão `"balanced"` pronta como o scikit-learn; este parâmetro multiplica o gradiente da classe positiva, e o teto no valor exato da proporção é o ponto em que a reponderação neutraliza o desbalanceamento por completo. |

**O que fica para os cards seguintes.** Este notebook não mede custo por ajuste nem escolhe `n_iter` para a `RandomizedSearchCV` real — isso depende do #185, que ainda está aberto. Quando ele decidir a biblioteca de Gradient Boosting, a seção 2 deste notebook e o espaço correspondente em `src/ensembles.py` perdem a metade descartada no mesmo commit.

## 4. Pipeline e linha de base do Gradient Boosting (card 11, #188)

O card 11 foi dividido para liberar o D3 inteiro para a busca de hiperparâmetros do #190: esta seção monta o `Pipeline` que encadeia o `ColumnTransformer` do contrato (`src/matriz.py`) ao `HistGradientBoostingClassifier`, treina a linha de base com hiperparâmetros padrão e mede o resultado com `avaliar` do card 05 (#241).

**A biblioteca é o `HistGradientBoostingClassifier`.** O #185 decidiu por ele em vez do `XGBClassifier`: é o gradient boosting do próprio scikit-learn, já fixado no `requirements.txt`, sem dependência extra para o Colab instalar. A seção 2 acima ainda mantém os dois espaços de busca vivos porque o #186 foi escrito antes dessa decisão; a limpeza do espaço descartado é commit do próprio #185, não deste card.

**Sobre a função `avaliar`.** O card 05 (#241) ainda não está em `develop`. Como o card #188 prevê para esse atraso, o pipeline e os testes unitários (`tests/test_ensembles.py`) já estão commitados com dados sintéticos e não dependem de `avaliacao.py`. Para esta seção registrar a linha de base **medida de verdade** em vez de um número sintético, a célula da subseção 4.4 traz uma cópia literal da função `avaliar` da branch do #241 — não uma reimplementação — e será substituída por `from avaliacao import avaliar` assim que aquele card chegar em `develop`.

### 4.1. Dependências e carga da base

Mesmo padrão de `regressao_logistica.ipynb`: a versão do `scikit-learn` precisa bater com a fixada em `requirements.txt`, porque é ela que determina o resultado do `HistGradientBoostingClassifier` medido abaixo. A base não é versionada, conforme o Termo de Abertura.

In [ ]:
!pip install -q -r requirements.txt

import sys
import time
from pathlib import Path

import pandas as pd
import sklearn

for pasta_codigo in [Path("src"), Path("../src"), Path("/content/src")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break

from ensembles import (SEMENTE_PADRAO, criar_pipeline_gradient_boosting,
                        medir_linha_de_base)
from matriz import preparar_matriz

print("scikit-learn", sklearn.__version__)

A versão impressa tem que ser `scikit-learn 1.9.1`, a mesma declarada em `requirements.txt` (chore #206). Divergência aqui invalida a comparação da seção 4.4 do artefato: o `HistGradientBoostingClassifier` de outra minor pode discretizar os histogramas de forma diferente e mudar o número medido nesta seção.

In [ ]:
# No Colab, monte o Drive antes de rodar esta célula:
#   from google.colab import drive; drive.mount('/content/drive')

CAMINHOS_POSSIVEIS = [
    Path("data/processed/base_analitica.parquet"),
    Path("../data/processed/base_analitica.parquet"),
    Path("/content/data/processed/base_analitica.parquet"),
    Path("/content/drive/MyDrive/projeto-azul/data/processed/base_analitica.parquet"),
]

caminho_base = next((c for c in CAMINHOS_POSSIVEIS if c.exists()), None)
if caminho_base is None:
    raise FileNotFoundError(
        "base_analitica.parquet não encontrada. Rode notebooks/pre-processamento.ipynb "
        f"ou ajuste o caminho. Procurado em: {[str(c) for c in CAMINHOS_POSSIVEIS]}"
    )

df = pd.read_parquet(caminho_base)
print(f"{caminho_base}: {df.shape[0]:,} linhas, {df.shape[1]} colunas")

### 4.2. Matriz do contrato

`preparar_matriz` é a única fonte de `X`, `y` e do pré-processador nesta seção, com as mesmas datas de corte do registro de decisão do #127 já usadas em `regressao_logistica.ipynb` — nenhum corte novo é decidido aqui.

In [ ]:
CORTE_VALIDACAO = "2025-07-01"
CORTE_TESTE = "2026-01-01"

inicio = time.perf_counter()
preparo = preparar_matriz(df, corte_validacao=CORTE_VALIDACAO, corte_teste=CORTE_TESTE)
print(f"preparar_matriz: {time.perf_counter() - inicio:.1f}s")

x_treino, y_treino = preparo["x"]["treino"], preparo["y"]["treino"]
x_validacao, y_validacao = preparo["x"]["validacao"], preparo["y"]["validacao"]
print(f"treino: {len(y_treino):,} linhas | validação: {len(y_validacao):,} linhas")

### 4.3. Montagem do pipeline (CR01)

`criar_pipeline_gradient_boosting`, de `src/ensembles.py`, encadeia o `ColumnTransformer` de `preparo["preprocessador"]` — **clonado**, nunca remontado — e o `HistGradientBoostingClassifier`. Sem `hiperparametros`, todo eixo de busca do #190 (`learning_rate`, `max_iter`, `max_leaf_nodes`, `l2_regularization`, `min_samples_leaf`) nasce no padrão da biblioteca; o único valor que a função força é `early_stopping=False`, pela mesma razão documentada em `modelo.HIPERPARAMETROS_CANDIDATO` (#103): o padrão `"auto"` separaria uma fatia aleatória do ajuste que ignora `ID_GOLDENRECORD`. `random_state=SEMENTE_PADRAO` (42) é explícito, não implícito.

In [ ]:
pipeline_gb = criar_pipeline_gradient_boosting(
    preparo["preprocessador"], random_state=SEMENTE_PADRAO,
)
pipeline_gb

**Leitura da saída.** Dois passos nomeados: `preparo` é o `ColumnTransformer` clonado do contrato (ainda não ajustado — o `clone` descarta o estado que `preparar_matriz` já tinha ajustado no treino inteiro), e `modelo` é o `HistGradientBoostingClassifier` com `early_stopping=False` e `random_state=42`, todo o resto no padrão da biblioteca.

### 4.4. Função `avaliar` do card 05 (#241)

Ver a nota da seção 4 sobre por que esta célula existe: é uma cópia literal, não uma reimplementação, da função publicada na branch do #241.

In [ ]:
"""Copia literal e temporaria de src/avaliacao.py, da branch
feat/#241/adiciona-funcao-avaliar-metricas-protocolo (card 05), ainda nao
mergeada em develop. Nao e uma reimplementacao: o corpo da funcao e identico ao
da branch de origem. Esta celula existe so para que a linha de base desta
secao seja medida com a metrica de verdade em vez de uma metrica sintetica, e
deve ser apagada e trocada por `from avaliacao import avaliar` assim que o
#241 chegar em develop, exatamente como o card #188 prevê no fallback do card
05.
"""
import warnings

import numpy as np
from sklearn.metrics import average_precision_score, fbeta_score, recall_score, roc_auc_score

BETA_F2 = 2

NOMES_METRICAS = ("F2", "Sensibilidade", "Precisão Média", "ROC-AUC")


def avaliar(y_true, y_pred, y_proba):
    """Calcula num dicionario unico as quatro metricas do protocolo (card 02A.3)."""
    y_true = np.asarray(y_true)

    if not np.any(y_true == 1):
        warnings.warn(
            "avaliar(): nenhum exemplo da classe positiva no lote avaliado "
            f"({len(y_true)} linha(s)); F2, Sensibilidade, Precisão Média e "
            "ROC-AUC retornam nan.",
            UserWarning,
            stacklevel=2,
        )
        return {nome: float("nan") for nome in NOMES_METRICAS}

    return {
        "F2": float(fbeta_score(y_true, y_pred, beta=BETA_F2, zero_division=0)),
        "Sensibilidade": float(recall_score(y_true, y_pred, zero_division=0)),
        "Precisão Média": float(average_precision_score(y_true, y_proba)),
        "ROC-AUC": float(roc_auc_score(y_true, y_proba)),
    }


### 4.5. Linha de base medida (CR02, CR04)

`medir_linha_de_base` ajusta o pipeline **uma vez** sobre `x_treino`/`y_treino` e pontua sobre `x_validacao`/`y_validacao` — a partição de validação que `preparar_matriz` já separou, não um recorte novo: nenhum `KFold`, `GroupKFold` ou `train_test_split` é instanciado nesta seção nem em `src/ensembles.py` (CR02). `avaliar` é chamada com os três vetores (`y_true`, `y_pred`, `y_proba`), e o dicionário de métricas volta exatamente como ela o devolveu.

In [ ]:
resultado = medir_linha_de_base(
    pipeline_gb, x_treino, y_treino, x_validacao, y_validacao, avaliar,
)
print(f"tempo total (ajuste + predição): {resultado['tempo_total_s']:.1f}s")
print(f"n_treino: {resultado['n_treino']:,} | n_avaliação: {resultado['n_avaliacao']:,}")
resultado["metricas"]

### 4.6. Registro da linha de base (CR04)

| Item | Valor |
|---|---|
| Biblioteca | `HistGradientBoostingClassifier` (scikit-learn), decidida pelo #185 |
| Versão | `scikit-learn 1.9.1` (`requirements.txt`, chore #206) |
| `random_state` | `42` (`ensembles.SEMENTE_PADRAO`) |
| Partição avaliada | validação (48.301 linhas), treino com 341.962 linhas |
| F2 | 0,2936 |
| Sensibilidade | 0,2556 |
| Precisão Média | 0,5176 |
| ROC-AUC | 0,7340 |
| Tempo total (ajuste + predição) | 8,9s |

Essa é a linha de base do card: o desempenho do Gradient Boosting **antes** de qualquer busca de hiperparâmetro, com o pré-processador do contrato e `early_stopping=False` (a única troca em relação ao padrão da biblioteca, pela razão de vazamento por Cliente da seção 4.3). É contra estes quatro números que a busca aleatória do #190 (`ESPACO_GRADIENT_BOOSTING_HISTGB`, seção 2.1 acima) precisa mostrar ganho; sem essa referência, qualquer resultado da busca pareceria bom por si só.

## 5. Busca aleatória do Gradient Boosting (#190)

O #188 deixou o `HistGradientBoostingClassifier` montado sobre o pré-processador do contrato, com os hiperparâmetros padrão da biblioteca, e registrou a linha de base da seção 4.6. Esta seção entrega esse mesmo pipeline a uma `RandomizedSearchCV`, que varre o espaço `ESPACO_GRADIENT_BOOSTING_HISTGB` do #186 (seção 2.1) e grava os hiperparâmetros vencedores num JSON versionado. A partir desse JSON, `ensembles.melhor_gradient_boosting()` remonta o vencedor para a dupla de Métricas e Decisões.

Três travas valem para tudo o que segue:

1. **A validação é agrupada por Cliente.** O mesmo `ID_GOLDENRECORD` aparece em vários voos. Os folds vêm de `validacao.criar_folds`, o splitter agrupado do contrato, e são conferidos por `validacao.conferir_folds` antes da busca. O `fit` recebe `groups=groups`. `cv` inteiro é recusado por `criar_busca_gradient_boosting`, pelo mesmo motivo do #189: num classificador, o scikit-learn converteria o inteiro num particionador estratificado que ignora o Cliente, e o vencedor seria o que mais memoriza pessoas.
2. **`n_iter = 40` e `random_state = 42` (CR01).** Quarenta é o mínimo do card, e a função recusa menos. A semente `42` (`SEMENTE_PADRAO`) fixa ao mesmo tempo o sorteio das 40 combinações e a semente de cada `HistGradientBoostingClassifier`, então duas execuções desta seção sorteiam as mesmas combinações e elegem o mesmo vencedor.
3. **O scoring é o oficial da dupla de Métricas e Decisões.** É o `scorer_f2` do #242 (F2 com `beta=2`, Detrator como classe positiva), o critério de busca que o protocolo do #238 fixou para as quatro duplas. A partição de teste não é tocada: a busca recebe só o treino, e a reconstrução da seção 5.3 é medida na validação, a mesma partição da linha de base.

### 5.1. Montagem da busca (CR01, CR02)

A célula abaixo materializa os cinco folds agrupados por Cliente sobre o treino cru, confere que nenhum Cliente vaza entre ajuste e validação e monta a `RandomizedSearchCV` com `busca_gradient_boosting.criar_busca_gradient_boosting`. Nada é ajustado ainda. `n_jobs=-1` distribui os ajustes inteiros (uma combinação num fold) entre os núcleos da sessão. A ordem em que eles terminam não muda nenhum número.

In [ ]:
import busca_gradient_boosting as busca_gb
from ensembles import ESPACO_GRADIENT_BOOSTING_HISTGB
from scorer_f2 import scorer_f2
from validacao import N_FOLDS, conferir_folds, criar_folds

N_ITER = 40
RANDOM_STATE_BUSCA = SEMENTE_PADRAO  # 42

groups = preparo["grupos"]["treino"]
folds_gb = criar_folds(x_treino, groups)
conferir_folds(folds_gb, groups)
print(f"folds do contrato: {len(folds_gb)}, conferidos contra o Cliente")

busca = busca_gb.criar_busca_gradient_boosting(
    preparo["preprocessador"], ESPACO_GRADIENT_BOOSTING_HISTGB, folds_gb, scorer_f2,
    n_iter=N_ITER, random_state=RANDOM_STATE_BUSCA, n_jobs=-1,
)
print(f"n_iter={busca.n_iter} | random_state={busca.random_state} | folds={len(busca.cv)} | scoring={busca.scoring}")
print(f"ajustes previstos: {N_ITER} x {N_FOLDS} + refit = {N_ITER * N_FOLDS + 1}")

**Leitura da saída.** A primeira linha confirma que os cinco folds saíram de `criar_folds` e passaram por `conferir_folds`: nenhum Cliente aparece em duas validações, nem no ajuste e na validação do mesmo fold. A segunda mostra a configuração que o CR01 pede, `n_iter=40` e `random_state=42`, os cinco folds do contrato no lugar de `cv` e o scorer F2 do #242 (`make_scorer(fbeta_score, beta=2, pos_label=1, zero_division=0)`). A terceira dá o custo da busca: 201 ajustes, 40 combinações vezes 5 folds mais o reajuste do vencedor no treino inteiro. Com os 8,9 s da linha de base (seção 4.6) por ajuste, a conta sequencial fica perto de 30 minutos. O paralelismo entre processos a reduz.

### 5.2. Execução da busca e artefatos versionados

`busca_gradient_boosting.executar_busca` confere os folds de novo, chama `busca.fit(x_treino, y_treino, groups=groups)` sobre a matriz **crua** do treino (quem transforma é o pipeline, dentro de cada fold) e mede o tempo total. `salvar_resultados` grava dois arquivos em `assets/`, os únicos desta seção que vão para o git:

- `hiperparametros_gradient_boosting.json`: os hiperparâmetros vencedores, o `random_state`, o `n_iter`, o número de folds, o melhor F2 médio e o tempo total. É o arquivo que `ensembles.melhor_gradient_boosting()` lê;
- `cv_resultados_gradient_boosting.json`: uma linha por combinação, só com parâmetros, F2 médio, desvio entre folds, posição no ranking e tempo médio de ajuste. Nenhum dado de Cliente entra.

Os dois são JSON, e não CSV, porque o `.gitignore` do projeto proíbe `*.csv` por compromisso com o parceiro. O modelo ajustado não é versionado, por tamanho.

In [ ]:
relato_busca = busca_gb.executar_busca(busca, x_treino, y_treino, groups)
caminho_hp, caminho_resumo = busca_gb.salvar_resultados(busca, relato_busca, random_state=RANDOM_STATE_BUSCA)

vencedores_gb = busca_gb.hiperparametros_vencedores(busca)
print("hiperparâmetros vencedores:")
for nome, valor in vencedores_gb.items():
    print(f"  {nome:18s} {valor}")
print(f"\nmelhor F2 médio nos folds: {relato_busca['melhor_score_medio']:.4f}")
print(f"tempo total da busca: {relato_busca['tempo_total_s']:.0f} s "
      f"({relato_busca['tempo_total_s'] / 60:.1f} min), {relato_busca['n_ajustes']} ajustes")
print("no limite do intervalo:", busca_gb.parametros_no_limite(vencedores_gb, ESPACO_GRADIENT_BOOSTING_HISTGB) or "nenhum")
print("\ngravados:", caminho_hp.name, "e", caminho_resumo.name)
busca_gb.resumir_cv_results(busca).head(10)

**Leitura da saída.** O bloco impresso traz os seis hiperparâmetros vencedores, o melhor F2 médio nos cinco folds, o tempo total e o número de ajustes. Esse número precisa ser 201: menos do que isso significaria um fold pulado ou uma combinação que falhou. A linha "no limite do intervalo" lista os vencedores que caíram na borda do espaço do #186. Um vencedor na borda diz que a busca queria ir além dela, e por isso é comentado na tabela da seção 5.4. A tabela no fim mostra as dez melhores combinações, com o desvio entre folds (`std_test_score`) ao lado da média. Uma diferença de F2 menor que esse desvio não separa dois candidatos.

### 5.3. Reconstrução pelo JSON e métrica oficial (CR04, CR05)

O F2 médio da seção 5.2 é o critério da busca, não o número que vai para a tabela comparativa. O número oficial é o de `avaliar`, do #241, e ele precisa sair de um modelo que qualquer pessoa consiga remontar sem esta sessão aberta. Por isso a célula abaixo não usa o `best_estimator_`. Ela chama `ensembles.melhor_gradient_boosting()`, a mesma função que a dupla de Métricas e Decisões vai importar, e essa função lê o JSON gravado em `assets/` e devolve o pipeline vencedor **não ajustado**, com o `random_state` do JSON.

O pipeline reconstruído passa por `medir_linha_de_base`, a mesma medição da seção 4.5: ajuste no treino, previsão na validação e `avaliar(y_true, y_pred, y_proba)`. Linha de base e vencedor ficam assim comparáveis número a número. `avaliar` agora vem de `src/avaliacao.py`, já em `develop`, e não da cópia da seção 4.4. Por fim, a célula mede também o `best_estimator_` da busca desta sessão e exige que as duas medições sejam iguais.

In [ ]:
from avaliacao import avaliar
from ensembles import PASSO_MODELO, melhor_gradient_boosting
from sklearn.exceptions import NotFittedError
from sklearn.utils.validation import check_is_fitted

melhor_gb = melhor_gradient_boosting(preparo["preprocessador"])
try:
    check_is_fitted(melhor_gb.named_steps[PASSO_MODELO])
    print("pipeline reconstruído já ajustado: não deveria")
except NotFittedError:
    print("pipeline reconstruído chega não ajustado")
print("random_state do reconstruído:", melhor_gb.named_steps[PASSO_MODELO].random_state)

resultado_melhor = medir_linha_de_base(melhor_gb, x_treino, y_treino, x_validacao, y_validacao, avaliar)
metricas_melhor = resultado_melhor["metricas"]

melhor_busca = busca.best_estimator_
metricas_busca = avaliar(y_validacao, melhor_busca.predict(x_validacao), melhor_busca.predict_proba(x_validacao)[:, 1])
assert metricas_busca == metricas_melhor, "o JSON não reconstrói o vencedor da busca"

comparacao = pd.DataFrame({
    "linha de base (#188)": resultado["metricas"],
    "vencedor da busca (#190)": metricas_melhor,
})
comparacao["diferença"] = comparacao["vencedor da busca (#190)"] - comparacao["linha de base (#188)"]
print(f"tempo do reconstruído (ajuste + predição): {resultado_melhor['tempo_total_s']:.1f} s")
print("idêntico ao best_estimator_ da busca: sim")
comparacao.round(4)

**Leitura da saída.** As duas primeiras linhas confirmam o CR04 sobre o objeto real: `melhor_gradient_boosting()` devolve o pipeline **sem ajuste**, com `random_state=42`, o mesmo do JSON. A linha "idêntico ao best_estimator_" só aparece se as quatro métricas do reconstruído forem exatamente iguais às do vencedor reajustado pela busca. Essa igualdade faz do JSON versionado uma descrição completa do vencedor. A tabela põe lado a lado as métricas de `avaliar` da linha de base da seção 4.6 e as do vencedor, na mesma partição de validação (48.301 linhas), e é dela que saem os números registrados na seção 5.4 (CR05).

### 5.4. Espaço de busca, vencedores e registro (CR01, CR03, CR05)

**Espaço do #186 lado a lado com o vencedor.** A coluna "No limite?" usa `parametros_no_limite`: um inteiro conta como borda só no piso ou no teto exatos, e um contínuo conta quando fica a menos de 5% da amplitude de um dos extremos (na escala log para o `learning_rate`).

| Hiperparâmetro | Espaço do #186 | Vencedor | No limite? |
| --- | --- | --- | --- |
| `learning_rate` | `loguniform` de 0,01 a 0,3 | 0,0498 | não |
| `max_leaf_nodes` | `randint` de 15 a 127 | 61 | não |
| `max_iter` | `randint` de 100 a 600 | 200 | não |
| `l2_regularization` | `uniform` de 0,0 a 2,0 | 1,7744 | não |
| `min_samples_leaf` | `randint` de 20 a 200 | 150 | não |
| `class_weight` | `None` ou `"balanced"` | `"balanced"` | não se aplica |
| `early_stopping` | fora do espaço, fixo em `False` pelo #188 | `False` | não se aplica |

**Registro da busca.**

| Item | Valor |
| --- | --- |
| `n_iter` | **40** (mínimo do card, recusado abaixo disso) |
| `random_state` | **42** (`SEMENTE_PADRAO`), no sorteio e no estimador |
| `cv` | 5 folds de `validacao.criar_folds`, agrupados por `ID_GOLDENRECORD`; `fit(..., groups=groups)` |
| `scoring` | `scorer_f2` do #242 (F2, `beta=2`, Detrator positivo) |
| Ajustes | 201 (40 × 5 + refit) |
| Melhor F2 médio nos folds | 0,5318 (desvio entre folds 0,0035) |
| Tempo total registrado | 7147 s de parede, `n_jobs=-1` em 12 núcleos, execução local (ver "Sobre o tempo" abaixo) |
| Hiperparâmetros versionados | `assets/hiperparametros_gradient_boosting.json` |
| Resumo das 40 combinações | `assets/cv_resultados_gradient_boosting.json` |

**Métrica do estimador reconstruído em `avaliar` (CR05).** Estes são os números que a célula da seção 5.3 imprime para `melhor_gradient_boosting()` ajustado no treino e medido na validação, ao lado da linha de base da seção 4.6.

| Métrica (`avaliar`, #241) | Linha de base (#188) | Vencedor reconstruído (#190) | Diferença |
| --- | --- | --- | --- |
| F2 | 0,2936 | 0,4974 | +0,2038 |
| Sensibilidade | 0,2556 | 0,5068 | +0,2511 |
| Precisão Média | 0,5176 | 0,5174 | −0,0002 |
| ROC-AUC | 0,7340 | 0,7330 | −0,0010 |

**Leitura.** O ganho de F2 na validação, de 0,2936 para 0,4974, vem quase todo de um eixo só, `class_weight`. As 21 combinações com `"balanced"` ficaram entre 0,512 e 0,532 de F2 médio nos folds, e as 19 sem reponderação ficaram entre 0,311 e 0,330, sem nenhuma sobreposição. Reponderar empurra a probabilidade predita para cima, e o corte padrão de 0,5 do `predict` passa a apontar mais Detratores. É isso que o F2, que pesa o recall quatro vezes mais que a precisão, premia: a Sensibilidade praticamente dobra (0,2556 para 0,5068). Precisão Média e ROC-AUC, que só dependem da ordenação do score, ficam onde estavam (−0,0002 e −0,0010).

Ou seja, a busca não produziu um modelo que **ordena** melhor os Clientes. Ela produziu um que **corta** num ponto mais favorável ao recall. Isso importa para a dupla de Métricas e Decisões. O limiar operacional por capacidade de contato (seção 4.3.2, card #247) vai redefinir esse corte de qualquer forma, e a comparação entre os quatro modelos deve ler Precisão Média e ROC-AUC ao lado do F2. Vale também a ressalva do #103: com `"balanced"`, a probabilidade predita deixa de refletir a frequência observada de 20,44% de Detratores e não pode ser lida diretamente como risco.

Entre as combinações balanceadas, as 13 melhores estão a menos de um desvio entre folds (0,0035) do vencedor. Os eixos numéricos quase não separam candidatos, e como nenhum vencedor ficou no limite do espaço, o intervalo do #186 não precisa ser ampliado. A linha de base reexecutada nesta sessão reproduziu exatamente os quatro números da seção 4.6.

**Sobre o tempo.** A busca rodou localmente, com `n_jobs=-1` em 12 núcleos, e não no Colab. No meio da execução a máquina ficou parada por cerca de 75 minutos, então os 7.147 s registrados superestimam o custo: o tempo efetivo de busca ficou perto de 45 minutos. Como referência de orçamento, vale o tempo médio de ajuste por combinação, que está em `cv_resultados_gradient_boosting.json` (93 s no vencedor, até 260 s nas combinações com `max_iter` acima de 500).

### 5.5. Como a dupla de Métricas e Decisões consome o vencedor

```python
from ensembles import melhor_gradient_boosting
from matriz import preparar_matriz

preparo = preparar_matriz(df, corte_validacao="2025-07-01", corte_teste="2026-01-01")
pipeline = melhor_gradient_boosting(preparo["preprocessador"])   # não ajustado, random_state=42
pipeline.fit(preparo["x"]["treino"], preparo["y"]["treino"])
```

A função não recebe hiperparâmetro nem semente: os dois vêm do JSON versionado, e é isso que garante que qualquer pessoa remonte o mesmo modelo. As travas que protegem a entrega rodam com `pytest tests/test_ensembles.py tests/test_busca_gradient_boosting.py -v`.

**Depende de:** #186 (espaço), #188 (pipeline e linha de base), #241 (`avaliar`) e #242 (`scorer_f2`). **Bloqueia:** #191, #192 e #194, que partem de `melhor_gradient_boosting()`.